In [ ]:
# =============================================================================
# jobs/37_rfg_exposure.py
# What lies under each Rapid Flood Guidance polygon, counted from the address points (as job 35 does for warning and alert areas).
#
# INPUT   config.TBL_RFG_UPDATES and TBL_RFG_POLYGONS (job 36), config.TBL_ADDRESS_POINTS (job 33), the warning and alert area boundaries,
#         and config.TBL_CRITICAL_SITES if it exists.
# OUTPUT  config.TBL_RFG_EXPOSURE          one row per polygon, with its is_valid flag: addresses, people, households, over 65s, people by rivers and sea band, by surface
#                                          water band, Flood Zone 3, the Census vulnerability counts. Written LAST: its rows mean the update is done.
#         config.TBL_RFG_CONSTITUENCIES    each polygon split by constituency (addresses, people, households, people in High or Medium)
#         config.TBL_RFG_AREAS             the people of each polygon who are inside each warning or alert area (a person in two areas appears in both)
#         config.TBL_RFG_SITES             the critical sites inside each polygon, with their rivers and sea band
# All restricted: they are built from address-level data. The polygon row is the distinct total; summing the area rows double counts overlaps.
#
# WHICH UPDATES  Those that gave guidance (is_present) with at least one polygon that have no rows in TBL_RFG_EXPOSURE yet. Set update_keys to redo
# particular ones. Every polygon whose geometry can be read is counted, valid or not: is_valid says whether the polygon is still current, and the polygons of past
# updates are no longer valid but are wanted for the history and for event review. Each output row carries is_valid, so the app can show what is in
# force now apart from the rest.
#
# HOW  The points are read one polygon at a time, in strips 5 km wide (easting), each strip only as high as the polygon, and the strips are added up (a point is
# in one strip only, so they add exactly). No read is large, so it runs on serverless as well as on a cluster. An update with polygons across England is
# read polygon by polygon, never by one box over all of them.
#
# WHAT TO CHECK (printed): polygons read and any left out; people under each polygon; the constituencies and areas it reaches.
# Widgets: update_keys.
# =============================================================================

import os
import sys

_nb_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
if not _nb_path.startswith("/Workspace"):
    _nb_path = "/Workspace" + _nb_path
PROJECT_ROOT = os.path.dirname(os.path.dirname(_nb_path))
sys.path.insert(0, PROJECT_ROOT)

def _have():
    try:
        import pyproj  # noqa: F401
        import shapely  # noqa: F401
        return True
    except ImportError:
        return False


if not _have():
    get_ipython().run_line_magic("pip", "install -q pyproj shapely")
    import importlib
    importlib.invalidate_caches()
    if not _have():
        print("Packages installed but not importable in this session: restarting Python. Run all again from the top.")
        dbutils.library.restartPython()

import importlib

import numpy as np
import pandas as pd
from pyspark.sql import functions as F

import config
import importlib as _il

_il.reload(config)   # a Python session that ran an earlier version of config.py would otherwise miss tables added since
import utils.area_exposure as _ae
import utils.rfg as _rfg
importlib.reload(_ae)
importlib.reload(_rfg)
from utils.area_exposure import to_bng_polygons
from utils.helpers import get_spark, table_exists, utc_now
from utils.rfg import EXPOSURE_COLS, merge_exposure, polygon_key, rfg_exposure

spark = get_spark()
dbutils.widgets.text("update_keys", "", "Only these updates (comma separated update_key). Blank is every in-force update not yet counted")
ONLY = [k.strip() for k in dbutils.widgets.get("update_keys").split(",") if k.strip()]


def show(df):
    display(df.astype(str))

In [ ]:
# =============================================================================
# 1. WHICH UPDATES, AND THEIR POLYGONS IN BRITISH NATIONAL GRID
# =============================================================================

upd = spark.table(config.TBL_RFG_UPDATES).filter("is_present AND n_polygons > 0").toPandas()
done = set()
if table_exists(spark, config.TBL_RFG_EXPOSURE):
    done = set(spark.table(config.TBL_RFG_EXPOSURE).select("update_key").distinct().toPandas()["update_key"])
todo = upd[upd["update_key"].isin(ONLY)] if ONLY else upd[~upd["update_key"].isin(done)]
print(f"Updates that gave guidance, with polygons: {len(upd)}; already counted: {len(done & set(upd['update_key']))}; to count now: {len(todo)}.")
if todo.empty:
    dbutils.notebook.exit("no_change")

keys_sql = ", ".join("'" + k.replace("'", "''") + "'" for k in todo["update_key"])
pol = spark.table(config.TBL_RFG_POLYGONS).filter(f"update_key IN ({keys_sql}) AND geometry IS NOT NULL").toPandas()
geoms, status = to_bng_polygons(pol["geometry"].tolist())
pol["geom_status"] = status
print("Polygons read, by geometry status:")
show(pol.groupby("geom_status").size().rename("polygons").reset_index())
pol = pol[pol["geom_status"] == "ok"].reset_index(drop=True)
polys = [g for g, s in zip(geoms, status) if s == "ok"]
pol["polygon_key"] = [polygon_key(k, n) for k, n in zip(pol["update_key"], pol["polygon_number"])]
VALID = dict(zip(pol["polygon_key"], pol["is_valid"].astype(bool)))

# the warning and alert areas, for "which areas does it reach"
frames = []
for table, kind in ((config.TBL_EA_FWA, "flood_warning"), (config.TBL_EA_FAA, "flood_alert")):
    t = spark.table(table).toPandas()
    t["kind"] = kind
    frames.append(t)
raw = pd.concat(frames, ignore_index=True).rename(columns={"ea_area_code": "area_id", "ea_area_name": "area_name"}).drop_duplicates(["kind", "area_id"])
a_geoms, a_status = to_bng_polygons(raw["geometry"].tolist())
ok = [s == "ok" for s in a_status]
areas = raw[ok].reset_index(drop=True)
a_polys = [g for g, s in zip(a_geoms, a_status) if s == "ok"]
print(f"Areas usable: {len(areas):,} of {len(raw):,}.")

sites = None
if spark.catalog.tableExists(config.TBL_CRITICAL_SITES):
    sites = spark.table(config.TBL_CRITICAL_SITES).select("site_id", "name", "site_type", "easting", "northing", "site_group", "rofrs_rank", "fz_rank").toPandas()
print(f"Critical sites: {0 if sites is None else len(sites):,}.")

In [ ]:
# =============================================================================
# 2. COUNT EACH UPDATE FROM THE ADDRESS POINTS INSIDE A BOX AROUND ITS POLYGONS
# =============================================================================

POINT_COLS = ["easting", "northing", "people", "households", "over65", "no_car_hh", "bad_health", "limited_lot", "imd_decile", "point_source", "pcon",
              "rofrs_rank", "rofsw_rank", "fz_rank", "fz_sea_rank", "fz_river_rank", "fz_river_and_sea_rank"]
points_t = spark.table(config.TBL_ADDRESS_POINTS)
cols = [c for c in POINT_COLS if c in points_t.columns]
ingested = pd.Timestamp(utc_now())
out = {"polygon": [], "constituency": [], "areas": [], "sites": []}

STRIP = 5000.0                                      # metres of easting per read
EMPTY = points_t.limit(0).select(*cols).toPandas()   # a points frame with no rows, for the sites-only call


def count_polygon(key_poly, poly):
    """One polygon, read in strips by easting and added up. The critical sites are found once, from a call with no points."""
    x0, y0, x1, y1 = poly.bounds
    parts = [rfg_exposure(EMPTY, [poly], [key_poly], a_polys, areas["area_id"].tolist(), areas["kind"].tolist(), areas["area_name"].tolist(), sites)]
    n_pts = 0
    for xs in np.arange(np.floor(x0 / STRIP) * STRIP, x1 + STRIP, STRIP):
        p = (points_t.filter((F.col("easting") >= float(max(xs, x0 - 1))) & (F.col("easting") < float(xs + STRIP)) & (F.col("easting") >= float(xs))
                             & (F.col("northing") >= float(y0)) & (F.col("northing") <= float(y1))).select(*cols).toPandas())
        if len(p):
            n_pts += len(p)
            parts.append(rfg_exposure(p, [poly], [key_poly], a_polys, areas["area_id"].tolist(), areas["kind"].tolist(), areas["area_name"].tolist(), None))
        del p
    return merge_exposure(parts), n_pts


for key in todo["update_key"]:
    idx = pol.index[pol["update_key"] == key].tolist()
    if not idx:
        print(f"Update {key}: no readable polygon. Skipped.")
        continue
    per_poly, n_total = [], 0
    for i in idx:
        r_i, n_i = count_polygon(pol.loc[i, "polygon_key"], polys[i])
        per_poly.append(r_i); n_total += n_i
    r = merge_exposure(per_poly)
    for name in out:
        f = r[name].copy()
        f["update_key"] = key
        f["is_valid"] = f["polygon_key"].map(VALID).fillna(False).astype(bool)
        out[name].append(f)
    tot = r["polygon"]["people"].sum()
    print(f"Update {key}: {len(idx)} polygons ({int(pol.loc[idx, 'is_valid'].sum())} still valid), {n_total:,} points read, {tot:,.0f} people under the polygons.")

res = {k: (pd.concat(v, ignore_index=True) if v else pd.DataFrame()) for k, v in out.items()}
poly_out = res["polygon"].drop(columns=["is_valid"], errors="ignore").merge(pol[["polygon_key", "polygon_number", "latest_update", "counties", "is_valid"]], on="polygon_key", how="left")
poly_out["counted_at"] = ingested
print("\nPeople under each polygon, largest first:")
show(poly_out.sort_values("people", ascending=False)[["update_key", "polygon_number", "counties", "addresses", "people", "households", "people_rs_high_medium"]].head(15))

In [ ]:
# =============================================================================
# 3. WRITE: the detail tables first, the polygon table last
# Rows for the counted updates are deleted first, so a re-run never duplicates; a failure part way leaves the update looking not counted.
# =============================================================================

counted = list(res["polygon"]["update_key"].unique()) if len(res["polygon"]) else []
in_list = ", ".join("'" + k.replace("'", "''") + "'" for k in counted)


def put(df, table, comment):
    if table_exists(spark, table) and counted:
        spark.sql(f"DELETE FROM {table} WHERE update_key IN ({in_list})")
    if len(df):
        spark.createDataFrame(df).write.format("delta").mode("append").option("mergeSchema", "true").saveAsTable(table)
    spark.sql(f"COMMENT ON TABLE {table} IS '{comment}'")
    print(f"{table}: {len(df):,} rows")


put(res["constituency"], config.TBL_RFG_CONSTITUENCIES, "Each Rapid Flood Guidance polygon split by constituency: addresses, people, households, people in a High or Medium rivers and sea band. Restricted. Job 37.")
put(res["areas"], config.TBL_RFG_AREAS, "The people of each Rapid Flood Guidance polygon who are inside each flood warning or alert area. A person in two areas of one kind is in both rows, so do not add them. Restricted. Job 37.")
put(res["sites"], config.TBL_RFG_SITES, "The critical sites inside each Rapid Flood Guidance polygon, with their rivers and sea band. Restricted. Job 37.")
put(poly_out, config.TBL_RFG_EXPOSURE, "What lies under each Rapid Flood Guidance polygon (is_valid says whether it is still current), counted from the address points: addresses, people, households, over 65s, people by rivers and sea band and surface water band, Flood Zone 3 and the Census vulnerability counts. Written last: its rows mean the update is counted. Restricted. Job 37.")

In [ ]:
# Exit is in its own cell: in an interactive run Databricks replaces a cell's output with the exit message,
# so the checks above would be hidden if this line shared their cell.
dbutils.notebook.exit("success")